# Organizar PNGs de simulaciones en carpetas GIF

Este notebook copia los archivos `.png` de cada subcarpeta `B-X.X` hacia tres carpetas:
- `Gif_mapa` → `mapa_corrientes_(N).png`
- `GIF_cooper` → `densidad_cooper_(N).png`
- `Gif_Vi` → `curva_VI_(N).png`

Los archivos originales **no se eliminan**.

In [1]:
import os
import shutil
import re
from pathlib import Path

In [2]:
# ─────────────────────────────────────────────
# CONFIGURACIÓN  ← ajusta solo esta celda
# ─────────────────────────────────────────────

# Ruta raíz donde están las carpetas N50_crestas (solo esta carpeta se procesará)
ROOT = Path(".")   # cambia a la ruta absoluta si es necesario, p.ej. Path("/home/usuario/simulaciones")

N_VALUES   = [50]                       # solo N50_crestas
POLARITIES = ["positivo", "negativo"]

# Mapeo: nombre original del png → (carpeta destino, prefijo de salida)
FILE_MAP = {
    "mapa_corrientes.png" : ("Gif_mapa",   "mapa_corrientes"),
    "densidad_cooper.png" : ("GIF_cooper",  "densidad_cooper"),
    "curva_VI.png"        : ("Gif_Vi",      "curva_VI"),
}

In [3]:
def get_b_value(folder_name: str) -> float:
    """Extrae el valor numérico de B desde el nombre 'B-X.X'."""
    match = re.match(r"B-([\d.]+)", folder_name)
    if match:
        return float(match.group(1))
    return float("inf")  # si no reconoce el formato, va al final


def organizar(root: Path, n_values: list, polarities: list, file_map: dict):
    for n in n_values:
        for polarity in polarities:
            base = root / f"N{n}_crestas" / polarity

            if not base.exists():
                print(f"[AVISO] No existe: {base}")
                continue

            # Crear carpetas destino
            dest_folders = {}
            for _, (folder_name, _) in file_map.items():
                dest = base / folder_name
                dest.mkdir(exist_ok=True)
                dest_folders[folder_name] = dest

            # Obtener subcarpetas B-X.X ordenadas por valor numérico
            b_folders = sorted(
                [d for d in base.iterdir() if d.is_dir() and d.name.startswith("B-")],
                key=lambda d: get_b_value(d.name)
            )

            if not b_folders:
                print(f"[AVISO] Sin carpetas B- en: {base}")
                continue

            print(f"\n{'='*60}")
            print(f"N{n}_crestas / {polarity}  →  {len(b_folders)} valores de B")
            print(f"{'='*60}")

            for idx, b_folder in enumerate(b_folders, start=1):
                for src_name, (folder_name, prefix) in file_map.items():
                    src_file = b_folder / src_name
                    if not src_file.exists():
                        print(f"  [FALTANTE] {src_file}")
                        continue

                    dst_name = f"{prefix}_({idx}).png"
                    dst_file = dest_folders[folder_name] / dst_name
                    shutil.copy2(src_file, dst_file)
                    print(f"  ✓ {b_folder.name}/{src_name}  →  {folder_name}/{dst_name}")

    print("\n✅ ¡Listo! Todos los archivos fueron copiados y renombrados.")


organizar(ROOT, N_VALUES, POLARITIES, FILE_MAP)


N50_crestas / positivo  →  13 valores de B
  ✓ B-0.0/mapa_corrientes.png  →  Gif_mapa/mapa_corrientes_(1).png
  ✓ B-0.0/densidad_cooper.png  →  GIF_cooper/densidad_cooper_(1).png
  ✓ B-0.0/curva_VI.png  →  Gif_Vi/curva_VI_(1).png
  ✓ B-0.1/mapa_corrientes.png  →  Gif_mapa/mapa_corrientes_(2).png
  ✓ B-0.1/densidad_cooper.png  →  GIF_cooper/densidad_cooper_(2).png
  ✓ B-0.1/curva_VI.png  →  Gif_Vi/curva_VI_(2).png
  ✓ B-0.2/mapa_corrientes.png  →  Gif_mapa/mapa_corrientes_(3).png
  ✓ B-0.2/densidad_cooper.png  →  GIF_cooper/densidad_cooper_(3).png
  ✓ B-0.2/curva_VI.png  →  Gif_Vi/curva_VI_(3).png
  ✓ B-0.3/mapa_corrientes.png  →  Gif_mapa/mapa_corrientes_(4).png
  ✓ B-0.3/densidad_cooper.png  →  GIF_cooper/densidad_cooper_(4).png
  ✓ B-0.3/curva_VI.png  →  Gif_Vi/curva_VI_(4).png
  ✓ B-0.4/mapa_corrientes.png  →  Gif_mapa/mapa_corrientes_(5).png
  ✓ B-0.4/densidad_cooper.png  →  GIF_cooper/densidad_cooper_(5).png
  ✓ B-0.4/curva_VI.png  →  Gif_Vi/curva_VI_(5).png
  ✓ B-0.5/mapa_corrie

## Verificación rápida

Corre la siguiente celda para ver un resumen de cuántos archivos quedaron en cada carpeta destino.

In [4]:
print(f"{'Ruta':<70} {'Archivos':>8}")
print("-" * 80)

gif_folders = ["Gif_mapa", "GIF_cooper", "Gif_Vi"]

for n in N_VALUES:
    for polarity in POLARITIES:
        for gf in gif_folders:
            path = ROOT / f"N{n}_crestas" / polarity / gf
            if path.exists():
                count = len(list(path.glob("*.png")))
                print(f"{str(path):<70} {count:>8}")
            else:
                print(f"{str(path):<70} {'NO EXISTE':>8}")

Ruta                                                                   Archivos
--------------------------------------------------------------------------------
N50_crestas/positivo/Gif_mapa                                                13
N50_crestas/positivo/GIF_cooper                                              13
N50_crestas/positivo/Gif_Vi                                                  13
N50_crestas/negativo/Gif_mapa                                                13
N50_crestas/negativo/GIF_cooper                                              13
N50_crestas/negativo/Gif_Vi                                                  13
